# 07b - Validación de preguntas de negocio con focused_chunked

Objetivos:
- comparar `enriched` versus `focused_chunked` en preguntas de negocio
- medir si el contexto recuperado es más específico y menos ruidoso
- dejar un CSV comparativo sin depender del LLM


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import OUTPUTS_DIR
from src.rag_chain import answer_question, build_context, retrieve_documents

TOP_K = 5
GENERATE_LLM_ANSWERS = False
OUTPUT_PATH = OUTPUTS_DIR / "tables" / "business_question_validation_focused_chunked.csv"


In [2]:
questions = [
    {
        "question_id": "biz_001",
        "category": "fit_proveedor",
        "question": "Mi empresa provee medicamentos hospitalarios. Que convocatorias recuperadas parecen mas alineadas y por que?",
    },
    {
        "question_id": "biz_002",
        "category": "fit_proveedor",
        "question": "Mi empresa vende reactivos de laboratorio. Que evidencia aparece en la convocatoria para pensar que si corresponde a nuestro rubro?",
    },
    {
        "question_id": "biz_003",
        "category": "fit_proveedor",
        "question": "Somos una empresa de software para gestion clinica. La convocatoria realmente pide software o solo equipamiento?",
    },
    {
        "question_id": "biz_004",
        "category": "resumen_requisitos",
        "question": "Resume los puntos clave que un proveedor deberia revisar antes de postular a esta convocatoria de alcantarillado.",
    },
    {
        "question_id": "biz_005",
        "category": "resumen_requisitos",
        "question": "Que insumos o materiales principales se solicitan realmente en esta convocatoria mas alla del titulo corto?",
    },
    {
        "question_id": "biz_006",
        "category": "decision_preliminar",
        "question": "Esta convocatoria parece viable para una empresa pequena o sugiere una carga documental y tecnica alta?",
    },
    {
        "question_id": "biz_007",
        "category": "decision_preliminar",
        "question": "Que senales del texto indican que esta oportunidad puede requerir requisitos formales, garantias o mayor experiencia previa?",
    },
    {
        "question_id": "biz_008",
        "category": "alineacion_rubro",
        "question": "Somos contratistas de obras sanitarias. Las convocatorias recuperadas realmente encajan con nuestro rubro o hay ruido en los resultados?",
    },
]

display(pd.DataFrame(questions))


,question_id,category,question
0,biz_001,fit_proveedor,Mi empresa provee medicamentos hospitalarios. ...
1,biz_002,fit_proveedor,Mi empresa vende reactivos de laboratorio. Que...
2,biz_003,fit_proveedor,Somos una empresa de software para gestion cli...
3,biz_004,resumen_requisitos,Resume los puntos clave que un proveedor deber...
4,biz_005,resumen_requisitos,Que insumos o materiales principales se solici...
5,biz_006,decision_preliminar,Esta convocatoria parece viable para una empre...
6,biz_007,decision_preliminar,Que senales del texto indican que esta oportun...
7,biz_008,alineacion_rubro,Somos contratistas de obras sanitarias. Las co...


In [3]:
scenarios = [
    {"scenario": "keyword_enriched", "retrieval_mode": "keyword", "corpus_variant": "enriched"},
    {"scenario": "hybrid_enriched", "retrieval_mode": "hybrid", "corpus_variant": "enriched"},
    {"scenario": "keyword_focused", "retrieval_mode": "keyword", "corpus_variant": "focused_chunked"},
    {"scenario": "hybrid_focused", "retrieval_mode": "hybrid", "corpus_variant": "focused_chunked"},
]

rows = []

for question_row in questions:
    question = question_row["question"]
    for scenario in scenarios:
        metadata_filters = {"corpus_variant": scenario["corpus_variant"]}
        sources = retrieve_documents(
            question,
            retrieval_mode=scenario["retrieval_mode"],
            k=TOP_K,
            metadata_filters=metadata_filters,
        )
        context = build_context(
            question,
            retrieval_mode=scenario["retrieval_mode"],
            k=TOP_K,
            metadata_filters=metadata_filters,
        )

        llm_answer = ""
        if GENERATE_LLM_ANSWERS:
            response = answer_question(
                question,
                retrieval_mode=scenario["retrieval_mode"],
                k=TOP_K,
                metadata_filters=metadata_filters,
            )
            llm_answer = response.answer

        rows.append(
            {
                "question_id": question_row["question_id"],
                "category": question_row["category"],
                "scenario": scenario["scenario"],
                "retrieval_mode": scenario["retrieval_mode"],
                "corpus_variant": scenario["corpus_variant"],
                "question": question,
                "retrieved_count": len(sources),
                "retrieved_cuces": " | ".join([str(item.get("cuce", "")) for item in sources if item.get("cuce")]),
                "retrieved_entidades": " | ".join([str(item.get("entidad", "")) for item in sources if item.get("entidad")]),
                "context_chars": len(context),
                "context_preview": context[:1200],
                "llm_answer": llm_answer,
            }
        )

results_df = pd.DataFrame(rows)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Resultados guardados en:", OUTPUT_PATH)
display(results_df[["question_id", "category", "scenario", "retrieved_count", "retrieved_cuces", "context_chars"]])


Resultados guardados en: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/tables/business_question_validation_focused_chunked.csv


,question_id,category,scenario,retrieved_count,retrieved_cuces,context_chars
0,biz_001,fit_proveedor,keyword_enriched,5,26-1276-00-1669611-1-1 | 26-1211-00-1669850-1-...,12000
1,biz_001,fit_proveedor,hybrid_enriched,5,26-1704-00-1668525-1-1 | 26-0902-43-1667332-1-...,12000
2,biz_001,fit_proveedor,keyword_focused,5,26-0132-00-1668364-1-1 | 26-1114-00-1668731-1-...,9682
3,biz_001,fit_proveedor,hybrid_focused,5,26-0903-19-1667462-1-1 | 26-0902-21-1668824-1-...,6685
4,biz_002,fit_proveedor,keyword_enriched,5,26-2303-00-1668697-1-1 | 26-1701-00-1665905-1-...,12000
5,biz_002,fit_proveedor,hybrid_enriched,5,26-0902-42-1667500-1-1 | 26-0418-07-1669590-1-...,12000
6,biz_002,fit_proveedor,keyword_focused,5,26-0132-00-1669693-1-1 | 26-1701-00-1669624-1-...,9723
7,biz_002,fit_proveedor,hybrid_focused,5,26-0418-07-1669590-1-1 | 26-0417-08-1669786-1-...,7906
8,biz_003,fit_proveedor,keyword_enriched,5,26-0046-38-1660991-1-1 | 26-0417-09-1669039-1-...,12000
9,biz_003,fit_proveedor,hybrid_enriched,5,26-0046-38-1660991-1-1 | 26-0035-09-1669025-1-...,12000
